# Tutorial: Reproducing the CUB CBM Experiment

This notebook reproduces the full pipeline for the bird-species Concept
Bottleneck Model (CBM) on the CUB-200-2011 sparrow minimal pair, from raw
images to a trained, evaluated model:

1. Encode every CUB image with CLIP (no filtering -- unlike emails, every
   image in the official split is used, unmodified)
2. Restrict the task to the Le Conte's vs. Savannah Sparrow minimal pair
   and 6 relevant concepts
3. Per-split standardize the embeddings
4. Train the concept extractor on every class *except* the sparrow pair
5. Train the label predictor on the sparrow pair's own training rows
6. Evaluate end-to-end, with classification reports at every stage
7. **Sanity check**: verify the freshly-encoded data matches the originally
   published dataset (`NWeak/cub-mirror` on HuggingFace)

**Out of scope**: this notebook does *not* touch the actual human user
study (participant responses, trust/confidence analysis, etc.) -- see
`SUPPLEMENTARY_ANALYSIS.md`. For the full written explanation of every
step below, see `cub/cub_preprocessing.md` and `supplementary_materials.tex`
(repo root).

Run this notebook from `cub/notebooks/`. **Step 1 (CLIP encoding) needs a
GPU and the raw CUB-200-2011 images under `cub/data/cub/`** (`CUB_200_2011/`
+ `class_attr_data_10/`) and takes a few minutes; everything after that is
fast.

In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC

sys.path.insert(0, str(Path("../scripts").resolve()))
import encode_clip  # noqa: E402 -- cub/scripts/encode_clip.py, reused rather than duplicated

2026-07-22 14:59:34.406 | DEBUG    | CQA.datasets:<module>:19 - Available datasets: {'chestmnist': <class 'CQA.datasets.dataset_classes.CHESTMINST_Dataset'>, 'cub': <class 'CQA.datasets.dataset_classes.CUBDataset'>, 'celeba': <class 'CQA.datasets.dataset_classes.CelebA'>, 'celeba_mini': <class 'CQA.datasets.dataset_classes.CelebAMini'>, 'celeba_original': <class 'CQA.datasets.dataset_classes.CelebAOriginal'>, 'cifar10': <class 'CQA.datasets.dataset_classes.Cifar10Custom'>, 'dermamnist': <class 'CQA.datasets.dataset_classes.DERMAMINST_Dataset'>, 'nih4': <class 'CQA.datasets.dataset_classes.NIHChestXray4Original'>, 'nih': <class 'CQA.datasets.dataset_classes.NIHChestXrayOriginal'>, 'shapes3d_mini': <class 'CQA.datasets.dataset_classes.SHAPES3DMini'>, 'shapes3d_original': <class 'CQA.datasets.dataset_classes.SHAPES3DOriginal'>, 'shapes3d': <class 'CQA.datasets.dataset_classes.SHAPES3D_Custom'>}


## 1. Encode every image with CLIP

Unlike emails (which filters the raw corpus before encoding), **CUB uses
every image in the official train/val/test split, unmodified** -- no
sampling, exclusion, or class-level filtering. Images go through CLIP's own
standard preprocessing (`Resize` -> `CenterCrop` -> `ToTensor` -> `Normalize`
with CLIP's published mean/std), obtained directly from `clip.load()`, and
are encoded with `ViT-L/14`. This reuses `encode_split`/`get_image_paths`
from `cub/scripts/encode_clip.py` directly, rather than duplicating that
logic here.

In [2]:
import clip

CLIP_MODEL = "ViT-L/14"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class Args:
    dataset = "cub"
    data_root = str(Path("../data/cub").resolve())
    clip_model = CLIP_MODEL
    device = DEVICE
    batch_size = 256
    num_workers = 8
    download = False

args = Args()
clip_model, preprocess = clip.load(args.clip_model, device=args.device)
clip_model.eval()
print(f"CLIP {CLIP_MODEL} loaded on {DEVICE}")

CLIP ViT-L/14 loaded on cuda


In [3]:
raw_embeddings, raw_concepts, raw_labels, raw_image_paths = {}, {}, {}, {}
for split in ["train", "val", "test"]:
    emb, concepts, labels, image_paths = encode_clip.encode_split(clip_model, preprocess, args, split)
    raw_embeddings[split] = emb
    raw_concepts[split] = concepts
    raw_labels[split] = labels
    raw_image_paths[split] = image_paths
    print(f"{split}: {emb.shape[0]} images -> {emb.shape[1]}-dim embeddings")

2026-07-22 14:59:42.404 | DEBUG    | CQA.datasets:get_dataset:31 - Getting dataset cub with kwargs {'split': 'train', 'root': '/home/nicola.debole/projects/user-study-CBMs/cub/data/cub', 'transform': Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    <function _convert_image_to_rgb at 0x7f01762c3560>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
), 'download': False}


2026-07-22 14:59:42.572 | DEBUG    | CQA.datasets:__init__:68 - /home/nicola.debole/projects/user-study-CBMs/cub/data/cub/dataset_info.json


2026-07-22 14:59:42.573 | DEBUG    | CQA.datasets:__init__:70 - Loading dataset cub from /home/nicola.debole/projects/user-study-CBMs/cub/data/cub


cub/train:   0%|          | 0/19 [00:00<?, ?it/s]

cub/train:   5%|▌         | 1/19 [00:02<00:52,  2.92s/it]

...

cub/test:  91%|█████████▏| 21/23 [00:21<00:01,  1.07it/s]

cub/test:  96%|█████████▌| 22/23 [00:22<00:00,  1.07it/s]

cub/test: 100%|██████████| 23/23 [00:22<00:00,  1.20it/s]

cub/test: 100%|██████████| 23/23 [00:22<00:00,  1.01it/s]

test: 5794 images -> 768-dim embeddings


## 2. Task restriction: the sparrow minimal pair

The CBM experiment does not use the full 200-way classification task.
Both the concept space and the label-prediction task are restricted to a
single, deliberately difficult *minimal pair*: the Le Conte's Sparrow
(class id 123) and the Savannah Sparrow (class id 126), and to six
concepts (of the 112 total) chosen for their relevance to distinguishing
these two species.

In [4]:
CONCEPT_MASK = [23, 44, 48, 69, 89, 103]  # striped breast, buff breast, white throat, buff nape, solid belly, brown crown
SPARROW_PAIR = [123, 126]  # Le Conte Sparrow, Savannah Sparrow

concept_names = encode_clip.load_names(Path("../metadata/cub/concepts.txt"))
class_names = encode_clip.load_names(Path("../metadata/cub/classes.txt"))
masked_concept_names = [concept_names[i] for i in CONCEPT_MASK]
print("Masked concepts:", masked_concept_names)
print("Sparrow pair:", [class_names[i] for i in SPARROW_PAIR])

Masked concepts: ['striped breast', 'buff breast', 'white throat', 'buff nape', 'solid belly', 'brown crown']
Sparrow pair: ['Le Conte Sparrow', 'Savannah Sparrow']


## 3. Per-split standardization

Each split's embeddings are independently standardized to zero mean and
unit variance, *per split* (using that split's own statistics, not
train-split statistics applied everywhere).

In [5]:
def standardize(embeddings):
    return (embeddings - embeddings.mean(0, keepdim=True)) / embeddings.std(0, keepdim=True)

train_embeddings = standardize(raw_embeddings["train"])
val_embeddings = standardize(raw_embeddings["val"])
test_embeddings = standardize(raw_embeddings["test"])

train_concepts, train_y = raw_concepts["train"], raw_labels["train"]
test_concepts, test_y = raw_concepts["test"], raw_labels["test"]

## 4. Train the concept extractor (excluding the sparrow pair)

**The concept extractor is trained on the training split with the two
sparrow classes excluded** -- it never sees a Le Conte's or Savannah
Sparrow training image, and is evaluated only on the held-out *test*-split
sparrow-pair images below. This ensures the concept predictions used at
evaluation time come from a model that generalizes the visual concepts
from unrelated species, rather than one fit directly to the two classes it
will later be asked to distinguish.

In [6]:
train_other_subset = np.where(~np.isin(train_y.numpy(), SPARROW_PAIR))[0]
test_pair_subset = np.where(np.isin(test_y.numpy(), SPARROW_PAIR))[0]

concept_model = MultiOutputClassifier(SVC(kernel="rbf", C=1.0, class_weight="balanced"))
concept_model.fit(
    train_embeddings[train_other_subset].numpy(),
    train_concepts[train_other_subset][:, CONCEPT_MASK].numpy(),
)
print(f"Concept extractor trained on {len(train_other_subset)} images (all classes except the sparrow pair).")

Concept extractor trained on 4745 images (all classes except the sparrow pair).


### Concept extractor: classification report on the sparrow pair's test images

In [7]:
X_test = test_embeddings[test_pair_subset].numpy()
y_test_concepts = test_concepts[test_pair_subset][:, CONCEPT_MASK].numpy()
concept_preds = concept_model.predict(X_test)
print(classification_report(y_test_concepts, concept_preds, target_names=masked_concept_names))

                precision    recall  f1-score   support

striped breast       0.54      0.97      0.69        30
   buff breast       0.67      0.97      0.79        29
  white throat       0.82      0.77      0.79        30
     buff nape       0.41      0.59      0.49        29
   solid belly       0.93      0.45      0.60        29
   brown crown       0.50      0.30      0.38        30

     micro avg       0.60      0.67      0.64       177
     macro avg       0.64      0.67      0.62       177
  weighted avg       0.64      0.67      0.62       177
   samples avg       0.63      0.67      0.63       177



## 5. Train the label predictor (only on the sparrow pair)

Trained *only* on the sparrow pair's own training rows (the images the
concept extractor above never saw), using ground-truth concepts rescaled
from {0,1} to {-1,+1}. Unlike emails, default regularization and a fitted
intercept are used -- with only two well-separated classes and 6
ground-truth concepts, the model reaches perfect training-time
separability regardless.

In [8]:
train_pair_subset = np.where(np.isin(train_y.numpy(), SPARROW_PAIR))[0]
train_pair_concepts_pm1 = 2 * train_concepts[train_pair_subset][:, CONCEPT_MASK].numpy() - 1
train_pair_y = train_y[train_pair_subset].numpy()

label_model = LogisticRegression(max_iter=1000, class_weight="balanced")
label_model.fit(train_pair_concepts_pm1, train_pair_y)
print(f"Label predictor trained on {len(train_pair_subset)} sparrow-pair images.")

Label predictor trained on 51 sparrow-pair images.


### Label predictor: classification report, evaluated directly on ground-truth test concepts

This is the accuracy **upper bound** -- bypassing the concept extractor entirely.

In [9]:
test_pair_y = test_y[test_pair_subset].numpy()
test_pair_concepts_pm1 = 2 * y_test_concepts - 1

y_pred_gt = label_model.predict(test_pair_concepts_pm1)
print(classification_report(test_pair_y, y_pred_gt))

              precision    recall  f1-score   support

         123       1.00      1.00      1.00        29
         126       1.00      1.00      1.00        30

    accuracy                           1.00        59
   macro avg       1.00      1.00      1.00        59
weighted avg       1.00      1.00      1.00        59



## 6. End-to-end evaluation

As with emails, the two stages are chained at inference time: embedding ->
per-concept SVM decision function -> `tanh` -> label predictor.

In [10]:
logits = np.column_stack([est.decision_function(X_test) for est in concept_model.estimators_])
concept_activations = np.tanh(logits)
y_pred_e2e = label_model.predict(concept_activations)

print(classification_report(test_pair_y, y_pred_e2e))
acc = accuracy_score(test_pair_y, y_pred_e2e)
print(f"End-to-end accuracy = {acc:.4f}")

              precision    recall  f1-score   support

         123       0.78      0.86      0.82        29
         126       0.85      0.77      0.81        30

    accuracy                           0.81        59
   macro avg       0.82      0.81      0.81        59
weighted avg       0.82      0.81      0.81        59

End-to-end accuracy = 0.8136


## 7. Sanity check: does the freshly-encoded data match the original?

This compares the *raw* (pre-standardization) embeddings/concepts/labels
just encoded above against the originally published `NWeak/cub-mirror`
dataset on HuggingFace, to confirm this notebook reproduces the same
encoding, not just "a" pipeline that happens to run. Joined on
`image_path`, which is stable across re-encoding (`sample_idx`/row order
is not guaranteed to be, since it depends on filesystem iteration order).

In [11]:
from datasets import load_dataset

hf = load_dataset("NWeak/cub-mirror")

mismatches = []
for split in ["train", "val", "test"]:
    if raw_image_paths[split] is None:
        mismatches.append(f"{split}: local image paths unavailable, cannot join for comparison")
        continue

    local_df_idx = {path: i for i, path in enumerate(raw_image_paths[split])}
    hf_df = hf[split].to_pandas().set_index("image_path")

    if set(local_df_idx) != set(hf_df.index):
        mismatches.append(f"{split}: image_path sets differ "
                           f"(local-only={len(set(local_df_idx) - set(hf_df.index))}, "
                           f"hf-only={len(set(hf_df.index) - set(local_df_idx))})")
        continue

    local_order = list(local_df_idx.keys())
    local_idx = [local_df_idx[p] for p in local_order]
    hf_df = hf_df.loc[local_order]

    local_emb = raw_embeddings[split][local_idx].numpy()
    hf_emb = np.stack(hf_df["embedding"].values)
    if not np.allclose(local_emb, hf_emb, atol=1e-3):
        mismatches.append(f"{split}: embeddings differ beyond tolerance "
                           f"(max abs diff {np.abs(local_emb - hf_emb).max():.2e})")

    local_concepts_arr = raw_concepts[split][local_idx].numpy()
    hf_concepts_arr = np.stack(hf_df["concepts"].values)
    if not np.array_equal(local_concepts_arr, hf_concepts_arr):
        mismatches.append(f"{split}: concepts differ ({(local_concepts_arr != hf_concepts_arr).sum()} cells)")

    local_labels_arr = raw_labels[split][local_idx].numpy()
    hf_labels_arr = hf_df["label"].values
    if not np.array_equal(local_labels_arr, hf_labels_arr):
        mismatches.append(f"{split}: label differs ({(local_labels_arr != hf_labels_arr).sum()} rows)")

    print(f"[{split}] {len(local_order)} images checked against NWeak/cub-mirror: OK")

assert not mismatches, "Freshly-encoded data does not match NWeak/cub-mirror:\n" + "\n".join(mismatches)
print("\nPASSED: freshly-encoded train/val/test data (image_path, embeddings, concepts, labels) "
      "matches the originally published NWeak/cub-mirror dataset.")

/home/nicola.debole/projects/user-study-CBMs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[train] 4796 images checked against NWeak/cub-mirror: OK
[val] 1198 images checked against NWeak/cub-mirror: OK


[test] 5794 images checked against NWeak/cub-mirror: OK

PASSED: freshly-encoded train/val/test data (image_path, embeddings, concepts, labels) matches the originally published NWeak/cub-mirror dataset.
